# Assignment No: 5 - SVM and Decision Tree on the Iris Dataset

**Name:** Atharv Shinde | **Class:** T.Y IT-A | **Batch:** D | **Roll No:** 33076 | **Subject:** DSML

**Aim:** Implement Support Vector Machine (SVM) and Decision Tree classifiers on the Iris dataset and compare their performance.

## 1. Import libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## 2. Load and prepare the dataset

Iris has 150 samples, 4 features (sepal length/width, petal length/width) and 3 classes (setosa, versicolor, virginica). The data is split 80% train / 20% test.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

## 3. Helper function to plot a confusion matrix

In [ ]:
def plot_cm(cm, model_name):
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=target_names,
        yticklabels=target_names,
    )
    plt.title(f"Confusion Matrix: {model_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()

## 4. Linear SVM

SVM is distance/margin based, so features are **scaled**. The scaler is fitted on the training data only, then applied to the test data (this avoids data leakage).

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_model = SVC(kernel="linear", C=1.0, random_state=42)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

print("================ Linear SVM Performance ================")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}\n")
print(classification_report(y_test, y_pred_svm, target_names=target_names))
plot_cm(confusion_matrix(y_test, y_pred_svm), "Linear SVM")

## 5. Decision Tree Classifier

Trees split on feature thresholds, so scaling is not needed (scale-invariant). The tree is trained on the original, unscaled data.

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)  # Scale-invariant
y_pred_dt = dt_model.predict(X_test)

print("================ Decision Tree Performance ================")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}\n")
print(classification_report(y_test, y_pred_dt, target_names=target_names))
plot_cm(confusion_matrix(y_test, y_pred_dt), "Decision Tree")

## 6. Comparative metrics analysis

In [ ]:
# Accuracy and number of misclassifications
acc_svm = accuracy_score(y_test, y_pred_svm)
acc_dt = accuracy_score(y_test, y_pred_dt)
errors_svm = np.sum(y_test != y_pred_svm)
errors_dt = np.sum(y_test != y_pred_dt)

# Macro F1-scores
_, _, f1_svm, _ = precision_recall_fscore_support(y_test, y_pred_svm, average="macro")
_, _, f1_dt, _ = precision_recall_fscore_support(y_test, y_pred_dt, average="macro")

print("=== METRIC-BY-METRIC COMPARISON ===\n")

# CONDITION 1: Accuracy comparison
if acc_dt > acc_svm:
    print(f"1. Accuracy: Decision Tree is better ({acc_dt:.4f} vs {acc_svm:.4f})")
elif acc_svm > acc_dt:
    print(f"1. Accuracy: Linear SVM is better ({acc_svm:.4f} vs {acc_dt:.4f})")
else:
    print(f"1. Accuracy: Both models perform equally ({acc_svm:.4f})")

# CONDITION 2: Misclassification comparison
if errors_dt < errors_svm:
    print(f"2. Error Count: Decision Tree is better with fewer errors ({errors_dt} vs {errors_svm})")
elif errors_svm < errors_dt:
    print(f"2. Error Count: Linear SVM is better with fewer errors ({errors_svm} vs {errors_dt})")
else:
    print(f"2. Error Count: Both models made equal errors ({errors_svm})")

# CONDITION 3: Macro F1-score comparison
if f1_dt > f1_svm:
    print(f"3. Macro F1-Score: Decision Tree is better ({f1_dt:.4f} vs {f1_svm:.4f})")
elif f1_svm > f1_dt:
    print(f"3. Macro F1-Score: Linear SVM is better ({f1_svm:.4f} vs {f1_dt:.4f})")
else:
    print(f"3. Macro F1-Score: Both models perform equally ({f1_svm:.4f})")

## 7. (Optional) Cross-validation check

The test set has only 30 samples, so one split can be misleading. 5-fold cross-validation gives a more reliable comparison. Remove this cell if your teacher wants only the original comparison.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

svm_cv = cross_val_score(
    make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0)), X, y, cv=5
)
dt_cv = cross_val_score(DecisionTreeClassifier(random_state=42), X, y, cv=5)

print(f"Linear SVM 5-fold CV accuracy   : {svm_cv.mean():.4f} (+/- {svm_cv.std():.4f})")
print(f"Decision Tree 5-fold CV accuracy: {dt_cv.mean():.4f} (+/- {dt_cv.std():.4f})")

## 8. Conclusion

- Both models classify Iris very well. *Setosa* is perfectly separated; the only confusion is between *versicolor* and *virginica*, whose petal measurements overlap.
- On this 80/20 split the unpruned Decision Tree scored 1.0000 accuracy (0 errors) and the Linear SVM scored 0.9667 (1 error: one versicolor predicted as virginica).
- A perfect score on only 30 test samples should be treated with care: an unpruned tree can overfit, so cross-validation (section 7) gives a fairer comparison.
- The Decision Tree is easy to interpret (rules), while the SVM gives a smooth maximum-margin boundary.